In [ ]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm


DATASET_DIR = "/kaggle/input/datasets/haliten/33halitsen-espcn-dataset"
TRAIN_HR_DIR = os.path.join(DATASET_DIR, "train/hr")
TRAIN_LR_DIR = os.path.join(DATASET_DIR, "train/lr")
VALID_HR_DIR = os.path.join(DATASET_DIR, "test/hr_1080p")
VALID_LR_DIR = os.path.join(DATASET_DIR, "test/lr_360p")

OUTPUT_DIR = "/kaggle/working/espcn_hybrid_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "espcn_original_y_only.pth")
CSV_SAVE_PATH = os.path.join(OUTPUT_DIR, "training_history_hybrid.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
UPSCALE_FACTOR = 3
BATCH_SIZE = 32

LEARNING_RATE = 2e-4 
MAX_EPOCHS = 100

EARLY_STOP_PATIENCE = 25 


class YChannelDataset(Dataset):
    def __init__(self, hr_dir, lr_dir):
        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.image_filenames = [f for f in os.listdir(hr_dir) if f.endswith(('.png', '.jpg'))]
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        hr_img = Image.open(os.path.join(self.hr_dir, img_name)).convert("YCbCr")
        lr_img = Image.open(os.path.join(self.lr_dir, img_name)).convert("YCbCr")

        y_hr, _, _ = hr_img.split()
        y_lr, _, _ = lr_img.split()

        return self.transform(y_lr), self.transform(y_hr)


class ESPCN_Y_Hybrid(nn.Module):
    def __init__(self, upscale_factor=3):
        super(ESPCN_Y_Hybrid, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 1 * (upscale_factor**2), kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(upscale_factor)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pixel_shuffle(self.conv3(x))
        return x

class Trainer:
    def __init__(self, model, train_loader, valid_loader):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.criterion = nn.L1Loss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=LEARNING_RATE)
        
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, 'min', factor=0.5, patience=10, threshold=1e-5)
        
        self.best_val_loss = float('inf')
        self.epochs_no_improve = 0
        self.best_model_wts = copy.deepcopy(self.model.state_dict())
        self.history = {"Epoch": [], "Train_Loss": [], "Valid_Loss": [], "LR": []}

    def train(self):
        print(f"[{DEVICE.type.upper()}] Hibrit ESPCN (Y-Kanalı) Eğitimi Başlıyor...")
        for epoch in range(MAX_EPOCHS):
            self.model.train()
            train_loss = 0.0
            for lr_y, hr_y in tqdm(self.train_loader, desc=f"Epoch {epoch+1}"):
                lr_y, hr_y = lr_y.to(DEVICE), hr_y.to(DEVICE)
                self.optimizer.zero_grad()
                loss = self.criterion(self.model(lr_y), hr_y)
                loss.backward()
                self.optimizer.step()
                train_loss += loss.item() * lr_y.size(0)
            
            avg_train_loss = train_loss / len(self.train_loader.dataset)
            
            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for lr_y, hr_y in self.valid_loader:
                    lr_y, hr_y = lr_y.to(DEVICE), hr_y.to(DEVICE)
                    val_loss += self.criterion(self.model(lr_y), hr_y).item() * lr_y.size(0)
            
            avg_val_loss = val_loss / len(self.valid_loader.dataset)
            curr_lr = self.optimizer.param_groups[0]['lr']
            print(f"-> Train: {avg_train_loss:.6f} | Valid: {avg_val_loss:.6f} | LR: {curr_lr}")

            self.history["Epoch"].append(epoch + 1)
            self.history["Train_Loss"].append(avg_train_loss)
            self.history["Valid_Loss"].append(avg_val_loss)
            self.history["LR"].append(curr_lr)
            
            self.scheduler.step(avg_val_loss)

            if avg_val_loss < self.best_val_loss - 1e-5:
                self.best_val_loss = avg_val_loss
                self.epochs_no_improve = 0
                self.best_model_wts = copy.deepcopy(self.model.state_dict())
            else:
                self.epochs_no_improve += 1
            
            if self.epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"\n🛑 early stoping {self.best_val_loss:.6f})")
                break

        self.model.load_state_dict(self.best_model_wts)
        torch.save(self.model.state_dict(), MODEL_SAVE_PATH)
        pd.DataFrame(self.history).to_csv(CSV_SAVE_PATH, index=False)


if __name__ == "__main__":
    train_loader = DataLoader(YChannelDataset(TRAIN_HR_DIR, TRAIN_LR_DIR), batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(YChannelDataset(VALID_HR_DIR, VALID_LR_DIR), batch_size=1)
    trainer = Trainer(ESPCN_Y_Hybrid(), train_loader, valid_loader)
    trainer.train()